# 04 — Damir: OCR, Business Parameters & Terms

**Google Colab notebook.** Runtime → *Change runtime type* → **GPU (T4 / L4 / A100)** before running.

Runs GPU OCR over invoices/receipts, scores it against **real transcriptions**, then checks business-parameter presence and extracts payment terms.

| | |
|---|---|
| **Inputs** | Drive: `datasets/ocr_multitype/`, `inputs/annotations/`, `inputs/images/` |
| **Outputs** | `ocr_outputs.csv`, `parameter_presence_results.csv`, `terms_extraction_results.csv`, metrics JSON |
| **Expected runtime** | ~40–70 min on a T4 |
| **Compute profile** | `colab_gpu` (generous — full data, pinned in the profile cell) |

### How results get back to the team
Everything is written to Google Drive by `colab_bootstrap.publish()`, into **both**:
- `outputs/damir/<kind>/` — the *latest* copy
- `runs/damir/<UTC-timestamp>/<kind>/` — an immutable archive, so re-running never
  silently destroys an earlier result

Tell the integrator (Hessam) when you're done; he copies from `outputs/` into the repo.

> **Before you run:** `MyDrive/DL2_InvoiceAI/` must already contain `code/` (the repo's `src/`,
> `scripts/`, and `colab_bootstrap.py`) and `inputs/`. If it doesn't, the bootstrap cell fails
> fast with a message telling you exactly what's missing.

### You have two evaluation sets, with very different strength

| | OCR Dataset (**primary**) | Batch CSVs (**secondary**) |
|---|---|---|
| Images | 973, pre-split | 5,201, of which **1,413 annotated** |
| Coverage | **100%** | ~27% |
| Text GT | per-box transcription (52,331) | page-level blob |
| Fields | company, date, address, total | invoice, items, subtotal, payment_instructions |

Report the **primary** numbers as your headline — they have full coverage and per-box text, so
CER/WER is meaningful. Use the batch subset as a secondary, real-full-page-invoice check, and
**always state the denominator** (only ~197 of the 750 manifest images have any GT at all).

### Don't reimplement the shared logic
`src/parameter_checker.py` and `src/terms_extraction.py` already exist and are unit-tested. Import
them. If you find a real bug, report it rather than forking the logic.

In [ ]:
# --- GPU check: stop here if this says "no GPU" ---------------------------------
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or
      "!! NO GPU. Runtime > Change runtime type > Hardware accelerator = GPU, then re-run.")
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
# --- Mount Drive + load the shared bootstrap -----------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/DL2_InvoiceAI"   # <-- change if your folder differs

import sys, os, shutil, json, time
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

_bs = Path(DRIVE_ROOT) / "code" / "colab_bootstrap.py"
assert _bs.exists(), (
    f"Missing {_bs}.\nUpload the repo's colab/colab_bootstrap.py into "
    f"{DRIVE_ROOT}/code/ and re-run this cell."
)
sys.path.insert(0, str(_bs.parent))
import colab_bootstrap as CB

root  = CB.mount_drive(DRIVE_ROOT)
paths = CB.setup_paths(root)
CB.install_deps("easyocr", "rapidfuzz", "pandas", "opencv-python-headless", "jiwer")
print("Drive root:", root)

In [ ]:
# --- Pin the generous Colab budget --------------------------------------------
os.environ["IIP_COMPUTE_PROFILE"] = "colab_gpu"
from src.compute_profile import get_profile

P = get_profile()
print(json.dumps(P, indent=2, default=str))

RUN_TS = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())   # one archive folder for this run
T0 = time.time()

In [ ]:
# --- Datasets: read straight from Google Drive (NO Kaggle token needed) ------
# Primary eval set. The secondary invoice check reads inputs/images/ + inputs/annotations/, which are already in Drive.
# Paths are resolved tolerantly: if a dataset was copied one level too deep
# (e.g. ocr_multitype/invoice/train/... instead of ocr_multitype/train/...),
# it is found anyway and a NOTE is printed. No re-upload needed.
DATA = paths.inputs / "datasets"

BASE = CB.resolve_dataset_root(DATA / "ocr_multitype", ['train/annotations', 'val/annotations', 'test/annotations'])

print(f"  ocr_multitype  -> {BASE}")
print(f"                    " f"{sum(1 for _ in BASE.rglob(chr(42)) if _.is_file()):,} files")

In [ ]:
# --- Load the primary GT (per-box transcriptions + entities) -----------------
import pandas as pd, numpy as np, cv2

def load(sp):
    out = []
    for ap in sorted((BASE/sp/"annotations").glob("*.json")):
        d = json.loads(ap.read_text(encoding="utf-8"))
        img = next((BASE/sp/"images").glob(ap.stem+".*"), None)
        if img:
            out.append({"file_id": d.get("file_id", ap.stem), "img": img,
                        "entities": d.get("entities", {}),
                        "text": "\n".join(b.get("text", "") for b in d.get("ocr_boxes", []))})
    return out
test = load("test")
print("primary eval docs:", len(test), "(100% have GT text + entities)")
print("sample GT text:\n", test[0]["text"][:300])

In [ ]:
# --- GPU OCR ------------------------------------------------------------------
import easyocr, time
reader = easyocr.Reader(["en"], gpu=True)

N = P.get("max_images_per_class") or len(test)
rows, t0 = [], time.time()
for r in test[:N]:
    res = reader.readtext(str(r["img"]), detail=1, paragraph=False)
    txt = "\n".join(t for _, t, _ in res)
    conf = float(np.mean([c for _, _, c in res])) if res else 0.0
    rows.append({"document_id": r["file_id"], "image_path": str(r["img"]),
                 "ocr_text": txt, "mean_confidence": round(conf, 4),
                 "n_boxes": len(res), "source": "ocr_dataset_test"})
ocr = pd.DataFrame(rows)
print(f"OCR'd {len(ocr)} docs in {time.time()-t0:.0f}s "
      f"({(time.time()-t0)/max(len(ocr),1):.2f}s/doc)")
ocr.head(2)

In [ ]:
# --- Score OCR against the real transcriptions (CER / WER) -------------------
import jiwer
gtmap = {r["file_id"]: r["text"] for r in test}

def clean(s):
    return " ".join(str(s).upper().split())

cers, wers = [], []
for r in ocr.itertuples():
    g, h = clean(gtmap.get(r.document_id, "")), clean(r.ocr_text)
    if not g:
        continue
    cers.append(jiwer.cer(g, h))
    wers.append(jiwer.wer(g, h))

ocr_metrics = {
    "n_scored": len(cers),
    "cer_mean": round(float(np.mean(cers)), 4) if cers else None,
    "cer_median": round(float(np.median(cers)), 4) if cers else None,
    "wer_mean": round(float(np.mean(wers)), 4) if wers else None,
    "wer_median": round(float(np.median(wers)), 4) if wers else None,
}
print(json.dumps(ocr_metrics, indent=2))
print("\nLower is better. CER ~0.1 = roughly 1 character in 10 wrong.")

In [ ]:
# --- Business-parameter presence (shared module - do not reimplement) --------
from src import parameter_checker as PC
print("parameter_checker exposes:", [x for x in dir(PC) if not x.startswith('_')][:15])

def check(text):
    """Adapt to whatever entry point the shared module provides."""
    for fn in ("check_parameters", "check_presence", "evaluate", "run"):
        if hasattr(PC, fn):
            try:
                return getattr(PC, fn)(text)
            except TypeError:
                pass
    # Fallback: presence of the four primary entity fields.
    t = text.upper()
    return {"has_company": any(k in t for k in ["LTD", "SDN", "BHD", "INC", "CO."]),
            "has_date": bool(__import__("re").search(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", t)),
            "has_total": "TOTAL" in t,
            "has_address": any(k in t for k in ["JALAN", "STREET", "ROAD", "NO."])}

pres = pd.DataFrame([{"document_id": r.document_id, **check(r.ocr_text)}
                     for r in ocr.itertuples()])
print(pres.drop(columns=["document_id"]).mean(numeric_only=True).round(3))

# Truth from the entities block, so presence can actually be scored.
ent = {r["file_id"]: r["entities"] for r in test}
truth = pd.DataFrame([{"document_id": k,
                       "gt_company": bool(v.get("company")), "gt_date": bool(v.get("date")),
                       "gt_address": bool(v.get("address")), "gt_total": bool(v.get("total"))}
                      for k, v in ent.items()])
pres = pres.merge(truth, on="document_id", how="left")
pres.to_csv("/content/out_parameter_presence_results.csv", index=False)
pres.head(3)

In [ ]:
# --- Terms extraction (shared module) ----------------------------------------
from src import terms_extraction as TE
print("terms_extraction exposes:", [x for x in dir(TE) if not x.startswith('_')][:15])

def extract(text):
    for fn in ("extract_terms", "extract", "run", "get_terms"):
        if hasattr(TE, fn):
            try:
                return getattr(TE, fn)(text)
            except TypeError:
                pass
    return {}

terms = []
for r in ocr.itertuples():
    out = extract(r.ocr_text)
    terms.append({"document_id": r.document_id,
                  **(out if isinstance(out, dict) else {"terms": str(out)})})
tdf = pd.DataFrame(terms)
tdf.to_csv("/content/out_terms_extraction_results.csv", index=False)
print(tdf.head(3))
print("\nnon-empty extraction rate:",
      round(float(tdf.drop(columns=['document_id']).notna().any(axis=1).mean()), 3))

In [ ]:
# --- SECONDARY eval: real full-page invoices (state the denominator!) --------
# The batch annotation CSVs and the 750 invoice images are already in Drive inputs/.
ann = sorted((paths.inputs / "annotations").glob("batch1_*.csv"))
sec_metrics = {"skipped": True, "reason": "no batch annotation CSVs in inputs/annotations/"}

if ann:
    gt2 = pd.concat([pd.read_csv(p) for p in ann], ignore_index=True)
    gt2["stem"] = gt2["File Name"].astype(str).str.replace(".jpg", "", regex=False).str.strip()
    gt2 = gt2.drop_duplicates("stem").set_index("stem")

    man = pd.read_csv(paths.inputs / "invoice_manifest.csv")
    have = man[man.document_id.isin(gt2.index)]
    print(f"manifest rows WITH GT: {len(have)} / {len(man)} ({len(have)/len(man):.1%})")

    M = min(len(have), 120)          # keep it bounded - this is a secondary check
    c2 = []
    for r in have.head(M).itertuples():
        ip = root / r.image_path
        if not ip.exists():
            continue
        res = reader.readtext(str(ip), detail=1, paragraph=False)
        hyp = clean("\n".join(t for _, t, _ in res))
        ref = clean(gt2.loc[r.document_id, "OCRed Text"])
        if ref:
            c2.append(jiwer.cer(ref, hyp))
        rows.append({"document_id": r.document_id, "image_path": r.image_path,
                     "ocr_text": hyp, "mean_confidence": None,
                     "n_boxes": len(res), "source": "invoice_batch1"})
    sec_metrics = {"skipped": False, "n_scored": len(c2),
                   "cer_mean": round(float(np.mean(c2)), 4) if c2 else None,
                   "denominator_note": f"only {len(have)} of {len(man)} manifest images have GT"}
print(json.dumps(sec_metrics, indent=2))

In [ ]:
# --- Provenance: the _run block makes cross-run model comparison possible ------
def run_block(**kw):
    """Stamp every metrics JSON with how it was produced, so local-CPU and Colab-GPU
    results can be charted against each other later."""
    b = {
        "profile": P.get("profile_name", "colab_gpu"),
        "device": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"),
        "epochs": P.get("epochs"), "imgsz": P.get("imgsz"), "batch": P.get("batch"),
        "wall_clock_sec": round(time.time() - T0, 1),
        "timestamp_utc": RUN_TS, "member": "damir",
    }
    b.update(kw)
    return b


In [ ]:
# --- Write the four contracted outputs ---------------------------------------
OUTD = Path("/content/out"); OUTD.mkdir(exist_ok=True)
pd.DataFrame(rows).to_csv(OUTD/"ocr_outputs.csv", index=False)
shutil.copyfile("/content/out_parameter_presence_results.csv",
                OUTD/"parameter_presence_results.csv")
shutil.copyfile("/content/out_terms_extraction_results.csv",
                OUTD/"terms_extraction_results.csv")

met = Path("/content/out/ocr_parameter_metrics.json")
met.write_text(json.dumps({
    "ocr_primary": ocr_metrics,
    "ocr_secondary_invoices": sec_metrics,
    "parameter_presence_rate": pres.drop(columns=["document_id"])
                                   .mean(numeric_only=True).round(4).to_dict(),
    "_run": run_block(model="easyocr-en", engine="easyocr",
                      n_train_images=None, eval_set="OCR Dataset test split (98, 100% GT)"),
}, indent=2), encoding="utf-8")
print(met.read_text()[:900])

In [ ]:
# --- Publish to Drive (latest + immutable archive) -----------------------------
# 
to_publish = [
    ("predictions", OUTD / "ocr_outputs.csv"),
    ("metrics", met),
]
for kind, src in to_publish:
    if src is None:
        continue
    p = Path(src)
    if not p.exists():
        print(f"  skip (not produced): {p}")
        continue
    CB.publish("damir", p, kind, paths=paths, run_timestamp=RUN_TS)

print("\nLatest ->", paths.outputs("damir"))
print("Archive ->", paths.run_dir("damir", timestamp=RUN_TS))

In [ ]:
# --- publish the two remaining CSVs + hand off -------------------------------
for f in ["parameter_presence_results.csv", "terms_extraction_results.csv"]:
    CB.publish("damir", OUTD/f, "predictions", paths=paths, run_timestamp=RUN_TS)

up = paths.inputs/"upstream"/"damir"; up.mkdir(parents=True, exist_ok=True)
for f in ["ocr_outputs.csv", "parameter_presence_results.csv", "terms_extraction_results.csv"]:
    shutil.copyfile(OUTD/f, up/f)
print("handed off ->", up)

## Report log — fill this in before you finish

Copy your answers into `presentation/member_reports/damir_report_log.md` in the repo (or paste
them to the integrator). This is the raw material for the group report and slide deck, so be
specific and **honest about what didn't work**.

1. Which OCR engine, and why (EasyOCR vs PaddleOCR vs Tesseract) — did you compare any?
2. Headline CER/WER on the primary set, plus how preprocessing changed them.
3. The secondary invoice eval: your number AND its denominator (~197 of 750 have GT).
4. Where parameter_checker / terms_extraction needed adapting, and any bug you found.
5. Which business parameters are hardest to detect and what that means for Pistac.io readiness.

Also note anything the next stage needs from you, and which figure you'd put on a slide.